## TSM Lines for Abers Plot Comparisons

#### Imports

In [ ]:
import sys, os
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

In [ ]:
import pandas as pd
import numpy as np
import scipy as sci
import matplotlib.pyplot as pl
import matplotlib.image as img
import subprocess
import pathlib
import pyvista as pv
import copy

import fenics_sz.utils
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)

In [ ]:
from fenics_sz.sz_problems.sz_slab import create_slab, plot_slab
from fenics_sz.sz_problems.sz_geometry import create_sz_geometry
from fenics_sz.sz_problems.sz_steady_dislcreep import SteadyDislSubductionProblem
from fenics_sz.sz_problems.sz_tdep_dislcreep import TDDislSubductionProblem
from fenics_sz.sz_problems.sz_params import default_params, allsz_params

In [ ]:
from fenics_sz.fluid_release.perple_x_integration import get_PT_data_from_tabs, plot_PT_data
import fenics_sz.fluid_release.get_PT_curves 

In [ ]:
from fenics_sz.fluid_release.workflow_functions import (in_domain, get_st_grid, get_interpolator, predict_h2o,
                                                          Cell, remove_rehydration, get_water_loss, sorted_water_loss_by_layer,
                                                          get_TSMstye_line)

#### Read Perple_X data

In [ ]:
DMM_data = get_PT_data_from_tabs('DMMdamp_25')
uvolcs_data = get_PT_data_from_tabs('upvolc_25')
lvolcs_data = get_PT_data_from_tabs('lovolc_25')
dikes_data = get_PT_data_from_tabs('dike_25')
gabbros_data = get_PT_data_from_tabs('gabbro_25')

In [ ]:
interps = []
interps.append(get_interpolator(uvolcs_data))
interps.append(get_interpolator(lvolcs_data))
interps.append(get_interpolator(dikes_data))
interps.append(get_interpolator(gabbros_data))
interps.append(get_interpolator(DMM_data))

### Run Workflow on Abers Examples

In [ ]:
resscale = 5.0
u_res = 100
h_serp = 2.0

#### Nicaragua

In [ ]:
dirname = "07_Nicaragua"
sz_dict = allsz_params[dirname]
sz_dict['sediment'] = "Carbonate41_25"
sz_dict

In [ ]:
nicaragua_water_loss, nicaragua_water_losses_and_depths, nicaragua_layer_losses, nicaragua_layer_losses_and_depths = get_TSMstye_line(sz_dict, h_serp, u_res, resscale, interps)

In [ ]:
fig = pl.figure(figsize=(20,10))
axi = fig.add_subplot(1,1,1)

img = pl.imread("figures/abers_nicaragua.png") # change figure
ip = axi.imshow(img)
axi.axis('off')
ax = axi.inset_axes([0.184,0.017,0.589,0.9]) # change locations of lower left and upper right of inset axes

ax.patch.set_alpha(0.5)


ax.plot(nicaragua_layer_losses["sed"], [row[1] for row in nicaragua_layer_losses_and_depths["sed"]], label = "sediments")
ax.plot(nicaragua_layer_losses["u_volcs"], [row[1] for row in nicaragua_layer_losses_and_depths["u_volcs"]], label = "upper_volcs")
ax.plot(nicaragua_layer_losses["l_volcs"], [row[1] for row in nicaragua_layer_losses_and_depths["l_volcs"]], label = "lower_volcs")
ax.plot(nicaragua_layer_losses["dikes"], [row[1] for row in nicaragua_layer_losses_and_depths["dikes"]], "p--", markersize=3.4, label = "dikes")
ax.plot(nicaragua_layer_losses["gabbros"], [row[1] for row in nicaragua_layer_losses_and_depths["gabbros"]], label = "gabbros")
ax.plot(nicaragua_layer_losses["mantle"], [row[1] for row in nicaragua_layer_losses_and_depths["mantle"]], "r--", label = "mantle")

ax.yaxis.set_inverted(True)  # inverted axis with autoscaling
ax.legend()
ax.xaxis()
ax.yaxis()
# ax.set_xticks(np.linspace(0,10,8), "Tg/MYr/m")



ax.set_xticks([])
ax.set_yticks([])
ax.spines['bottom'].set_color('red')
ax.spines['top'].set_color('red')
ax.spines['right'].set_color('red')
ax.spines['left'].set_color('red')
ax.spines['bottom'].set_linewidth(1)
ax.spines['top'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

#### Alaska Peninsula

In [ ]:
dirname = "01_Alaska_Peninsula"
sz_dict = allsz_params[dirname]
sz_dict['sediment'] = "AlaskaTurb46_25"
sz_dict

In [ ]:
ak_pen_water_loss, water_losses_and_depths, ak_pen_layer_losses, ak_pen_layer_losses_and_depths = get_TSMstye_line(sz_dict, h_serp, u_res, resscale, interps)

In [ ]:
fig = pl.figure(figsize=(20,10))
axi = fig.add_subplot(1,1,1)

img = pl.imread("figures/abers_alaskan_peninsula.png") # change figure
ip = axi.imshow(img)
axi.axis('off')
ax = axi.inset_axes([0.189,0.02,0.438,0.9]) # change locations of lower left and upper right of inset axes

ax.patch.set_alpha(0.5)


ax.plot(ak_pen_layer_losses["sed"], [row[1] for row in ak_pen_layer_losses_and_depths["sed"]], label = "sediments")
ax.plot(ak_pen_layer_losses["u_volcs"], [row[1] for row in ak_pen_layer_losses_and_depths["u_volcs"]], label = "upper_volcs")
ax.plot(ak_pen_layer_losses["l_volcs"], [row[1] for row in ak_pen_layer_losses_and_depths["l_volcs"]], label = "lower_volcs")
ax.plot(ak_pen_layer_losses["dikes"], [row[1] for row in ak_pen_layer_losses_and_depths["dikes"]], "p--", label = "dikes")
ax.plot(ak_pen_layer_losses["gabbros"], [row[1] for row in ak_pen_layer_losses_and_depths["gabbros"]], label = "gabbros")
ax.plot(ak_pen_layer_losses["mantle"], [row[1] for row in ak_pen_layer_losses_and_depths["mantle"]], "r--", label = "mantle")

ax.yaxis.set_inverted(True)  # inverted axis with autoscaling
ax.legend()
ax.xaxis()
ax.yaxis()
# ax.set_xticks(np.linspace(0,10,8), "Tg/MYr/m")



ax.set_xticks([])
ax.set_yticks([])
ax.spines['bottom'].set_color('red')
ax.spines['top'].set_color('red')
ax.spines['right'].set_color('red')
ax.spines['left'].set_color('red')
ax.spines['bottom'].set_linewidth(1)
ax.spines['top'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

fig.savefig(output_folder / "abers_alaskan_peninsula_comparison")

#### North Honshu

In [ ]:
dirname = "48_N_Honshu"
sz_dict = allsz_params[dirname]
sz_dict['sediment'] = "Diatom44_25"
sz_dict

In [ ]:
honshu_water_loss, honshu_water_losses_and_depths, honshu_layer_losses, honshu_layer_losses_and_depths = get_TSMstye_line(sz_dict, h_serp, u_res, resscale, interps)

In [ ]:
fig = pl.figure(figsize=(20,10))
axi = fig.add_subplot(1,1,1)

img = pl.imread("figures/abers_honshu.png") # change figure
ip = axi.imshow(img)
axi.axis('off')
ax = axi.inset_axes([0.209,0.024,0.47,0.85]) # change locations of lower left and upper right of inset axes

ax.patch.set_alpha(0.5)


ax.plot(honshu_layer_losses["sed"], [row[1] for row in honshu_layer_losses_and_depths["sed"]], label = "sediments")
ax.plot(honshu_layer_losses["u_volcs"], [row[1] for row in honshu_layer_losses_and_depths["u_volcs"]], label = "upper_volcs")
ax.plot(honshu_layer_losses["l_volcs"], [row[1] for row in honshu_layer_losses_and_depths["l_volcs"]], label = "lower_volcs")
ax.plot(honshu_layer_losses["dikes"], [row[1] for row in honshu_layer_losses_and_depths["dikes"]], "p--", label = "dikes")
ax.plot(honshu_layer_losses["gabbros"], [row[1] for row in honshu_layer_losses_and_depths["gabbros"]], label = "gabbros")
ax.plot(honshu_layer_losses["mantle"], [row[1] for row in honshu_layer_losses_and_depths["mantle"]], "r--", label = "mantle")

ax.yaxis.set_inverted(True)  # inverted axis with autoscaling
ax.legend()
ax.xaxis()
ax.yaxis()
# ax.set_xticks(np.linspace(0,10,8), "Tg/MYr/m")



ax.set_xticks([])
ax.set_yticks([])
ax.spines['bottom'].set_color('red')
ax.spines['top'].set_color('red')
ax.spines['right'].set_color('red')
ax.spines['left'].set_color('red')
ax.spines['bottom'].set_linewidth(1)
ax.spines['top'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

fig.savefig(output_folder / "abers_n_honshu_comparison")

#### Cascadia

In [ ]:
dirname = "04_Cascadia"
sz_dict = allsz_params[dirname]
sz_dict['sediment'] = "Pelagic45_25"
sz_dict

In [ ]:
cascadia_water_loss, cascadia_water_losses_and_depths, cascadia_layer_losses, cascadia_layer_losses_and_depths = get_TSMstye_line(sz_dict, h_serp, u_res, resscale, interps)

In [ ]:
fig = pl.figure(figsize=(20,10))
axi = fig.add_subplot(1,1,1)

img = pl.imread("figures/abers_cascades.png") # change figure
ip = axi.imshow(img)
axi.axis('off')
ax = axi.inset_axes([0.18,0.017,0.363,0.869]) # change locations of lower left and upper right of inset axes

ax.patch.set_alpha(0.5)


ax.plot(cascadia_layer_losses["sed"], [row[1] for row in cascadia_layer_losses_and_depths["sed"]], label = "sediments")
ax.plot(cascadia_layer_losses["u_volcs"], [row[1] for row in cascadia_layer_losses_and_depths["u_volcs"]], label = "upper_volcs")
ax.plot(cascadia_layer_losses["l_volcs"], [row[1] for row in cascadia_layer_losses_and_depths["l_volcs"]], label = "lower_volcs")
ax.plot(cascadia_layer_losses["dikes"], [row[1] for row in cascadia_layer_losses_and_depths["dikes"]], "p--", label = "dikes")
ax.plot(cascadia_layer_losses["gabbros"], [row[1] for row in cascadia_layer_losses_and_depths["gabbros"]], label = "gabbros")
ax.plot(cascadia_layer_losses["mantle"], [row[1] for row in cascadia_layer_losses_and_depths["mantle"]], "r--", label = "mantle")

ax.yaxis.set_inverted(True)  # inverted axis with autoscaling
ax.legend()
ax.xaxis()
ax.yaxis()
# ax.set_xticks(np.linspace(0,10,8), "Tg/MYr/m")



ax.set_xticks([])
ax.set_yticks([])
ax.spines['bottom'].set_color('red')
ax.spines['top'].set_color('red')
ax.spines['right'].set_color('red')
ax.spines['left'].set_color('red')
ax.spines['bottom'].set_linewidth(1)
ax.spines['top'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

fig.savefig(output_folder / "abers_cascadia_comparison")

Vast differences in sediments in alaska peninsula and cascadia cases. I checked the input files for the sediments against what Geoff used since that's where the discrepancies lie, and they seemed fine

To do:
* get function to output data for plotting a tsm lite
* workflow for (G)WR calculations
* look into sediments, edit the json files

* finish the comparison
* TSM lite
* Global TSM
* water retention code

## TSM 'Lite'

In [ ]:
fig, ax = pl.subplots()

ax.plot(ak_pen_layer_losses["mantle"], [row[1] for row in ak_pen_layer_losses_and_depths["mantle"]], label = "Alaska Peninsula")
ax.plot(cascadia_layer_losses["mantle"], [row[1] for row in cascadia_layer_losses_and_depths["mantle"]], label = "Cascadia")
ax.plot(nicaragua_layer_losses["mantle"], [row[1] for row in nicaragua_layer_losses_and_depths["mantle"]], label = "Nicaragua")
ax.plot(honshu_layer_losses["mantle"], [row[1] for row in honshu_layer_losses_and_depths["mantle"]], label = "North Honshu")

ax.yaxis.set_inverted(True)
ax.set_title("TSM Lite")
ax.set_xlabel('slab H20 loss (Tg/MYr/m)')
ax.set_ylabel('depth (km)')
ax.legend()
pl.show()
fig.savefig(output_folder / ("TSM_lite_v1"))